In [0]:
from pyspark.sql.functions import *

# Read all the data from volume idp/default/final_project
docs_df = spark.read.format("binaryFile").load("/Volumes/idp/default/final_project")
# display(docs_df)

# Parse each document using ai_parse_document (use expr for sql function)
parsed_df = docs_df.withColumn('parsed_content', expr("ai_parse_document(content)"))
parsed_df = parsed_df.drop('content')
# display(parsed_df)

# Save the parsed content to a delta table
spark.sql("create schema if not exists idp.processed")
parsed_df.write.format("delta").mode("overwrite").saveAsTable("idp.processed.docs_parsed")

In [0]:
# Reading from parsed delta table
parsed_df = spark.read.table("idp.processed.docs_parsed")
# display(parsed_df)

# Processing parsed text using ai_query
ENDPOINT = "databricks-gpt-oss-20b"

# Example prompt for LLM
prompt_prefix = '''
You are a helpful assistant. Given a JSON object representing a parsed document (with pages, elements, and metadata), convert the content into clean, readable markdown. Use "== page ==" to separate each page. Preserve important structure such as headers, tables, and captions. Do not include any JSON or code blocks in the output-just the clean markdown text.

JSON:
'''

# Apply ai_query to batch process the parsed JSON text
transformed_df = (
    parsed_df.withColumn(
        "clean_markdown_text",
        expr(f"""
         ai_query(  
            '{ENDPOINT}', 
            CONCAT('{prompt_prefix}', CAST(parsed_content AS STRING)), 
            responseFormat => '{{"type": "text"}}'
            )
        """)
    )
)   

display(transformed_df.select("path", "clean_markdown_text"))

# The above method is bit expensive method as it is using LLM to process the json, the cheapest way is to convert using Fast Plain text conversion, where we rapidly concatenate all text elements from the parsed JSON into a single plain text string. (sample code is in final_project.sql file)


path,clean_markdown_text
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"== page == # Riverbend Marketing Studio 22 Kingfisher Ave Nashville, TN 37203 (615) 555-0172 • invoices@riverbendstudio.com Website: riverbendstudio.com ## INVOICE - **Invoice #:** INV-20250010 - **Invoice Date:** 2025-01-17 - **Due Date:** 2025-01-17 - **Terms:** Due on Receipt - **PO #:** PO-13352 - **Currency:** USD - **Payment Method:** Wire Transfer | Bill To | Ship To | |---------|---------| | | | Keystone Logistics Partners 3400 River Rd Harrisburg, PA 17110 | Description | Qty | Unit Price | Amount | |-------------|-----|------------|--------| | Creative design (hours) | 8 | $93.48 | $747.84 | | Ad spend management (hours) | 12 | $78.47 | $941.64 | | Landing page build | 1 | $1,159.52 | $1,159.52 | | Performance report | 1 | $385.23 | $385.23 | | Campaign setup | 1 | $1,326.56 | $1,326.56 | | Remit To: Riverbend Marketing StudioPO Box 1903Nashville, TN 37202 | Subtotal | | $4,560.79 | | | Tax (0%) | | $0.00 | | | Total | | $4,560.79 | **Notes:** Please include the invoice number with your payment. For questions, contact Accounts Receivable."
dbfs:/Volumes/idp/default/final_project/invoice_16234.pdf,"== page == # Northstar IT Services LLC 410 Market St, Suite 1200 Philadelphia, PA 19106 (215) 555-0198 • billing@northstarit.com Website: northstarit.com ## INVOICE - **Invoice #:** INV-20250001 - **Invoice Date:** 2025-01-01 - **Due Date:** 2025-01-01 - **Terms:** Due on Receipt - **PO #:** PO-17238 - **Currency:** USD - **Payment Method:** Wire Transfer ### Bill To Ship To **Pinecrest Manufacturing Inc.** 1200 Industrial Pkwy, Pinedale, MA 02460 **Ship To:** 1200 Industrial Pkwy Allentown, PA | Description | Qty | Unit Price | Amount | |-------------|-----|------------|--------| | Dashboard maintenance | 10 | $71.59 | $715.90 | | User training session | 1 | $684.33 | $684.33 | | Data pipeline support (hourly) | 5 | $112.79 | $563.95 | | Cloud cost review | 2 | $131.48 | $262.96 | | Monthly retainer | 1 | $3,733.25 | $3,733.25 | | Remit To: Northstar IT Services LLCPO Box 8842Philadelphia, PA 19101 | Subtotal | | $5,960.39 | | | Tax (7%) | | $417.23 | | | Total | | $6,377.62 | **Notes:** Please include the invoice number with your payment. For questions, contact Accounts Receivable."
dbfs:/Volumes/idp/default/final_project/invoice_23433.pdf,"== page == # Northstar IT Services LLC 410 Market St, Suite 1200 Philadelphia, PA 19106 (215) 555-0198 • billing@northstarit.com Website: northstarit.com ## INVOICE - **Invoice #:** INV-20250002 - **Invoice Date:** 2025-01-15 - **Due Date:** 2025-01-15 - **Terms:** Due on Receipt - **PO #:** PO-15436 - **Currency:** USD - **Payment Method:** Wire Transfer ### Bill To Ship To Hawthorne Hospitality Group: 88 Westlake Blvd t/m 88Forward LLC/88Group: 88 Westlake Blvd t/m Richmond, VA 232 | Description | Qty | Unit Price | Amount | |-------------|-----|------------|--------| | Data pipeline support (hourly) | 6 | $93.55 | $561.30 | | Dashboard maintenance | 8 | $75.11 | $600.88 | | Cloud cost review | 10 | $101.29 | $1,012.90 | | Monthly retainer | 1 | $4,088.23 | $4,088.23 | | User training session | 1 | $1,143.35 | $1,143.35 | | Remit To: Northstar IT Services LLCPO Box 8842Philadelphia, PA 19101 | Subtotal | | $7,406.66 | | | Tax (0%) | | $0.00 | | | Total | | $7,406.66 | Notes: Please include the invoice number with your payment. For questions, contact Accounts Receivable."
dbfs:/Volumes/idp/default/final_project/invoice_34552.pdf,"== page == # Northstar IT Services LLC 410 Market St, Suite 1200 Philadelphia, PA 19106 (215) 555-0198 • billing@northstarit.com Website: northstarit.com ## INVOICE - **Invoice #:** INV-20250003 - **Invoice Date:** 2025-01-13 - **Due Date:** 2025-02-12 - **Terms:** Net 30 - **PO #:** PO-15763 - **Currency:** USD - **Payment Method:** Check ### Bill To / Ship To | Bill To | Ship To | |---------|---------| | | | ### Address Pinecrest Manufacturing Inc. 1200 Industrial Pkwy, Pine

In [0]:
pip install langchain-text-splitters

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from pyspark. sql.types import StructType, StructField, StringType 
import pandas as pd

# Set chunking parameters: chunk_size controls the max length of each chunk, and chunk_overlap allows for overlapping
# text between chunks to improve retrieval quality.
CHUNK_SIZE = 200
CHUNK_OVERLAP = 20
# Build the text splitter with preferred separators.
# This splitter will break the text at page markers or other natural boundaries, preserving document structure where
# possible.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators= ["\n== page ==\n", "== page ==", "\n\n", "\n", "", ""]
)             

# Define the output schema for the chunked DataFrame.
# Each row will contain the document path and a single text chunk.
schema = StructType([
StructField( "path", StringType(), True), StructField("chunk", StringType(), True),
])

def split_rows (iterator):
    """
    mapInPandas function: input pdfs with columns [path, clean_markdown_text], output rows [path, chunk].
    This function splits each document's text into chunks and yields them for DataFrame construction.
    """     
    for pdf in iterator:
        out = []
        for _, row in pdf.iterrows():
            path = row ["path"]
            text = row["clean_markdown_text"]
            if isinstance(text, str) and text. strip() : 
                for c in splitter.split_text(text) :
                    if c and c.strip():
                        out. append ((path, c))
        yield pd.DataFrame(out, columns=["path", "chunk"])

# Apply the splitter to the plain text DataFrame.
# This step transforms each document into multiple chunked rows for efficient downstream retrieval and embedding.
df_chunks = (
    transformed_df
    .select ("path", "clean_markdown_text")
    .mapInPandas (split_rows, schema=schema)
)

# Display the resulting chunked DataFrame for inspection.
display(df_chunks)



path,chunk
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"== page == # Riverbend Marketing Studio 22 Kingfisher Ave Nashville, TN 37203 (615) 555-0172 • invoices@riverbendstudio.com Website: riverbendstudio.com ## INVOICE"
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,- **Invoice #:** INV-20250010 - **Invoice Date:** 2025-01-17 - **Due Date:** 2025-01-17 - **Terms:** Due on Receipt - **PO #:** PO-13352 - **Currency:** USD
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,- **Payment Method:** Wire Transfer
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"| Bill To | Ship To | |---------|---------| | | | Keystone Logistics Partners 3400 River Rd Harrisburg, PA 17110"
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,| Description | Qty | Unit Price | Amount | |-------------|-----|------------|--------| | Creative design (hours) | 8 | $93.48 | $747.84 | | Ad spend management (hours) | 12 | $78.47 | $941.64 |
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"| Landing page build | 1 | $1,159.52 | $1,159.52 | | Performance report | 1 | $385.23 | $385.23 | | Campaign setup | 1 | $1,326.56 | $1,326.56 |"
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"| Remit To: Riverbend Marketing StudioPO Box 1903Nashville, TN 37202 | Subtotal | | $4,560.79 | | | Tax (0%) | | $0.00 | | | Total | | $4,560.79 |"
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"Notes: Please include the invoice number with your payment. For questions, contact Accounts Receivable."
dbfs:/Volumes/idp/default/final_project/invoice_16234.pdf,"== page == # Northstar IT Services LLC 410 Market St, Suite 1200 Philadelphia, PA 19106 (215) 555-0198 • billing@northstarit.com Website: northstarit.com ## INVOICE"
dbfs:/Volumes/idp/default/final_project/invoice_16234.pdf,## INVOICE Invoice #: INV-20250001 Invoice Date: 2025-01-01 Due Date: 2025-01-01 Terms: Due on Receipt PO #: PO-17238 Currency: USD Payment Method: Wire Transfer ### Bill To Ship To


In [0]:
from pyspark. sql import functions as F

# Add a unique, incremental id column before saving
df_chunks = df_chunks.withColumn("id", F.monotonically_increasing_id())

# Save the chunked data with id to the Delta table for retrieval and embedding
df_chunks.write.format ("delta").mode( "overwrite"). option("mergeSchema", "true"). saveAsTable("idp.processed.docs_chunked")

display(spark.read.table("idp.processed.docs_chunked"))

path,chunk,id
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"== page == # Riverbend Marketing Studio 22 Kingfisher Ave Nashville, TN 37203 (615) 555-0172 • invoices@riverbendstudio.com Website: riverbendstudio.com ## INVOICE",0
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,- **Invoice #:** INV-20250010 - **Invoice Date:** 2025-01-17 - **Due Date:** 2025-01-17 - **Terms:** Due on Receipt - **PO #:** PO-13352 - **Currency:** USD,1
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,- **Payment Method:** Wire Transfer,2
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"| Bill To | Ship To | |---------|---------| | | | **Keystone Logistics Partners** 3400 River Rd Harrisburg, PA 17110",3
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,| Description | Qty | Unit Price | Amount | |-------------|-----|------------|--------| | Creative design (hours) | 8 | $93.48 | $747.84 | | Ad spend management (hours) | 12 | $78.47 | $941.64 |,4
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"| Landing page build | 1 | $1,159.52 | $1,159.52 | | Performance report | 1 | $385.23 | $385.23 | | Campaign setup | 1 | $1,326.56 | $1,326.56 |",5
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"| **Remit To:** Riverbend Marketing StudioPO Box 1903Nashville, TN 37202 | **Subtotal** | | $4,560.79 | | | **Tax (0%)** | | $0.00 | | | **Total** | | $4,560.79 |",6
dbfs:/Volumes/idp/default/final_project/invoice_10119.pdf,"**Notes:** Please include the invoice number with your payment. For questions, contact Accounts Receivable.",7
dbfs:/Volumes/idp/default/final_project/invoice_16234.pdf,"== page == # Northstar IT Services LLC 410 Market St, Suite 1200 Philadelphia, PA 19106 (215) 555-0198 • billing@northstarit.com Website: northstarit.com ## INVOICE",8
dbfs:/Volumes/idp/default/final_project/invoice_16234.pdf,- **Invoice #:** INV-20250001 - **Invoice Date:** 2025-01-01 - **Due Date:** 2025-01-01 - **Terms:** Due on Receipt - **PO #:** PO-17238 - **Currency:** USD,9
